In [1]:
import pandas as pd
from pymongo import MongoClient

In [2]:
#=======================================================
# 1. csv 파일 읽어오기
#=======================================================
file = "./data/sales_data.csv"

df = pd.read_csv(file)

print("데이터 확인")
df


데이터 확인


,date,region,product,sales,profit
0,2024-01-01,인천,A,675,155
1,2024-01-02,광주,A,921,29
2,2024-01-02,부산,D,143,96
3,2024-01-02,서울,A,383,116
4,2024-01-02,광주,C,119,155
...,...,...,...,...,...
995,2024-12-29,부산,C,170,102
996,2024-12-29,서울,D,955,167
997,2024-12-30,광주,B,997,163
998,2024-12-31,부산,C,869,192


In [6]:
#=======================================================
# 2. MongoDB 연결
#=======================================================
client = MongoClient("mongodb://localhost:27017/")

# MongoDB 데이터베이스 생성
db = client["sales_db"]

# 컬렉션 생성 (>> RDBMS Table)
collection = db["sales"]

In [7]:
#=======================================================
# 3. 기존 데이터 삭제
#=======================================================
collection.delete_many({})
print("기존 mongodb 데이터 삭제 완료")

기존 mongodb 데이터 삭제 완료


In [8]:
#=======================================================
# 4. DataFrame --> MongoDB 저장
#=======================================================
data = df.to_dict("records")

collection.insert_many(data)

print("데이터 저장 완료")

데이터 저장 완료


In [9]:
#=======================================================
# 5. MongoDB에서 데이터 읽어오기
#=======================================================
for document in collection.find().limit(5):
    print(document)

{'_id': ObjectId('6aace711fb90704ee88ac4e9'), 'date': '2024-01-01', 'region': '인천', 'product': 'A', 'sales': 675, 'profit': 155}
{'_id': ObjectId('6aace711fb90704ee88ac4ea'), 'date': '2024-01-02', 'region': '광주', 'product': 'A', 'sales': 921, 'profit': 29}
{'_id': ObjectId('6aace711fb90704ee88ac4eb'), 'date': '2024-01-02', 'region': '부산', 'product': 'D', 'sales': 143, 'profit': 96}
{'_id': ObjectId('6aace711fb90704ee88ac4ec'), 'date': '2024-01-02', 'region': '서울', 'product': 'A', 'sales': 383, 'profit': 116}
{'_id': ObjectId('6aace711fb90704ee88ac4ed'), 'date': '2024-01-02', 'region': '광주', 'product': 'C', 'sales': 119, 'profit': 155}


In [12]:
#=======================================================
# 6. 지역별 평균 매출 계산
#=======================================================
# Mongodb Aggregation Pipeline
pipeline = [
    # region 값을 기준으로 그룹화
    { 
        "$group": {
            "_id": "$region",
            
            # 각 지역의 sales 평균 계산
            "avg_sales": {
                "$avg": "$sales"
            }
        }
    },
    {
        # 지역 이름 기준 정렬
        "$sort": {
            "_id": 1
        }
    }
]

# aggregation 실행
results = collection.aggregate(pipeline)


In [13]:
#=======================================================
# 7. 결과 출력
#=======================================================
print("\n====== 지역별 평균 매출액 ======")

for result in results:
    region = result["_id"]
    avg_sales = result["avg_sales"]

    print(
        f"지역: {region:5s} | "
        f"평균 매출액: {avg_sales:,.2f}"
)
    


====== 지역별 평균 매출액 ======
지역: 광주    | 평균 매출액: 555.29
지역: 대구    | 평균 매출액: 559.48
지역: 부산    | 평균 매출액: 525.20
지역: 서울    | 평균 매출액: 583.81
지역: 인천    | 평균 매출액: 546.68


In [14]:
## 연결 종료
client.close()
print("\nmongodb 연결 종료")


mongodb 연결 종료
